# EmpowerLens — D27 tests on Linux + GPU timing (Kaggle)

Two rows of the D27 test sheet that need Kaggle:

- **E-03 (Linux):** the full `pytest -q` suite on Kaggle's Linux image, to compare with the Windows count.
- **E-01 (one GPU):** a 50-reflection cohort report (binary decision + types + sentence selection + aggregation) on
  the D25 app models, timed end to end with `src.time_cohort_report`. Pass = under 300 seconds.

**Before running**
1. Settings → **Accelerator: GPU (T4 or P100)**, **Internet: On**.
2. **Add Input → Your Work → the D25 notebook** ("kaggle_runner_d25_app_models"), so its `app_models/` output
   (with `MODEL_CARD.json`) is mounted under `/kaggle/input/`. No HF token is needed: everything loads from there.

Takes about 10 minutes. Results are printed and saved to `/kaggle/working/d27/` (download `d27.zip`).

In [ ]:
# 1. Clone the repo into /kaggle/temp and install.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt pytest openpyxl
!git log --oneline -1 && python -c "import torch; print('cuda:', torch.cuda.is_available())"

In [ ]:
# 2. E-03: full test suite on Linux.
import subprocess, pathlib, platform
OUT = pathlib.Path("/kaggle/working/d27"); OUT.mkdir(parents=True, exist_ok=True)
r = subprocess.run("python -m pytest -q", shell=True, capture_output=True, text=True)
log = f"$ python -m pytest -q   ({platform.platform()})\n" + r.stdout + r.stderr
(OUT / "E-03_pytest_linux.txt").write_text(log)
print(log[-3000:])

In [ ]:
# 3. E-01: time a 50-reflection cohort report on the GPU with the D25 app models.
import glob, zipfile
cards = glob.glob("/kaggle/input/**/MODEL_CARD.json", recursive=True)
if not cards:   # only the zip was mounted: unzip it to /kaggle/temp
    zips = glob.glob("/kaggle/input/**/app_models.zip", recursive=True)
    assert zips, "D25 output not found: Add Input -> Your Work -> the D25 notebook"
    zipfile.ZipFile(zips[0]).extractall("/kaggle/temp/d25")
    cards = glob.glob("/kaggle/temp/d25/**/MODEL_CARD.json", recursive=True)
MODELS = os.path.dirname(cards[0])
print("models:", MODELS)
!python -m src.time_cohort_report --models "$MODELS" --n 50 --device cuda --out /kaggle/working/d27/E-01_gpu_timing.json

In [ ]:
# 4. Summary + zip for download.
import json
t = json.loads((OUT / "E-01_gpu_timing.json").read_text())
print(f"E-01: {t['seconds_total']} s total on {t['gpu']} (load {t['seconds_load_models']} s, report {t['seconds_report']} s)"
      f" -> {'PASS' if t['seconds_total'] < 300 else 'FAIL'} (target 300 s)")
print("E-03:", [l for l in (OUT / "E-03_pytest_linux.txt").read_text().splitlines() if " passed" in l or " failed" in l][-1:])
!cd /kaggle/working && rm -f d27.zip && zip -qr d27.zip d27 && ls -la d27